In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/14 15:41:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder = "../../../data/25.06/"
path_to_intermediate_data_folder = "../../../data/intermediate_files/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

disease_index_path = path_to_release_folder + "output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path = path_to_release_folder + "output/evidence/sourceId=chembl"
chembl_evidence = session.spark.read.parquet(platform_chembl_evidence_path)

all_evidence = session.spark.read.parquet(path_to_release_folder + "output/evidence")


efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids="MONDO_0045024",
)

chembl_evidence.show(1)

l2g_full = session.spark.read.parquet(path_to_intermediate_data_folder + "l2g_full_for_enrichment/")
l2g_full.count()

l2g_full.show(1)


g_p_s = session.spark.read.parquet(path_to_intermediate_data_folder + "genes_therapeutic_areas")
g_p_s.count()

26/08/14 15:41:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+----------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+----------+------------+-----------+--------------+-------------+----+------------------------+-----------------+-----------------------

8285

# Processing


In [4]:
rg_raw = session.spark.read.parquet(
    path_to_intermediate_data_folder + "canonical_pairwise_table/canonical_pairwise_table.parquet"
)

In [5]:
rg_raw.count()

625521

In [6]:
rg_raw.show()

+------------+------------+--------+---------+-----------+-----------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+----------+--------+-----------+-----------------------+---------------+-------------+------------------+--------------------+----------+--------+-----------+-----------------------+---------------+-------------+--------------------+-------------------+
|    studyId1|    studyId2|ancestry|runStatus|skip_reason|n_snps_used|                  rg|               rg_se|rg_clipped|                h2_1|             h2_1_se|                h2_2|             h2_2_se|                gcov|             gcov_se|           intercept|        intercept_se|            M_ldsc|   traitFromSource_1|nSamples_1|nCases_1|nControls_1|ldPopulationStructure_1|analysisFlags_1|  diseaseId_

In [7]:
si.df.show(1)

+--------------------+-----------+---------+--------------------+------------------------+-------------+------+---------------------+-----------+--------+----------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+---------+---------------------+-------------------+------------------+---------------+-------------+--------------------+-----------+---------+---------------+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|   diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|publicationTitle|publicationFirstAuthor|publicationDate|publicationJournal|backgroundTraitFromSourceMappedIds|backgroundDiseaseIds|   initialSampleSize|nCases|nControls|nSamples|  cohorts|ldPopulationStructure|   discoverySamples|replicationSamples|qualityControls|analysisFlags|summarystatsLocation|hasSumstats|condition|sumstatQCValues|
+-------------

In [8]:
si_df_clean = si.df.select("studyId", "diseaseIds")

df_final = (
    rg_raw.join(si_df_clean.alias("si1"), f.col("studyId1") == f.col("si1.studyId"), how="left")
    .join(si_df_clean.alias("si2"), f.col("studyId2") == f.col("si2.studyId"), how="left")
    .select(*rg_raw.columns, f.col("si1.diseaseIds").alias("diseaseIds1"), f.col("si2.diseaseIds").alias("diseaseIds2"))
    .drop("diseaseId_1", "diseaseId_2")
)

df_final.show()

+------------+------------+--------+---------+-----------+-----------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+----------+--------+-----------+-----------------------+---------------+------------------+--------------------+----------+--------+-----------+-----------------------+---------------+--------------------+-------------------+-------------+--------------------+
|    studyId1|    studyId2|ancestry|runStatus|skip_reason|n_snps_used|                  rg|               rg_se|rg_clipped|                h2_1|             h2_1_se|                h2_2|             h2_2_se|                gcov|             gcov_se|           intercept|        intercept_se|            M_ldsc|   traitFromSource_1|nSamples_1|nCases_1|nControls_1|ldPopulationStructure_1|analysisFlags_1|thera

In [9]:
# how ambiguous is the mapping?
(df_final.select(f.size("diseaseIds1").alias("n")).groupBy("n").count().orderBy("n").show())

+---+------+
|  n| count|
+---+------+
|  1|583579|
|  2| 38055|
|  3|  3887|
+---+------+



In [10]:
# fan-out factor
df_final.select(
    f.sum(f.size("diseaseIds1") * f.size("diseaseIds2")).alias("rows_after_explode"),
    f.count("*").alias("rows_now"),
).show()

+------------------+--------+
|rows_after_explode|rows_now|
+------------------+--------+
|            719314|  625521|
+------------------+--------+



In [11]:
# 1 ── cross product of the two id arrays
exploded = df_final.withColumn("diseaseId1", f.explode("diseaseIds1")).withColumn(
    "diseaseId2", f.explode("diseaseIds2")
)

# 2 ── drop self-pairs (same disease reached via two different studies)
self_pairs = exploded.filter(f.col("diseaseId1") == f.col("diseaseId2"))
print("self-pairs dropped:", self_pairs.count())
pairs = exploded.filter(f.col("diseaseId1") != f.col("diseaseId2"))
pairs.count()

self-pairs dropped: 75


719239

In [12]:
from pyspark.sql import functions as f

canon = pairs.withColumn("lo", f.least("diseaseId1", "diseaseId2")).withColumn(
    "hi", f.greatest("diseaseId1", "diseaseId2")
)

dups = canon.groupBy("lo", "hi").count().filter("count > 1")

n_dups = dups.count()
print(f"Duplicate/reversed pairs: {n_dups}")
dups.show()

Duplicate/reversed pairs: 50363


+-----------+-------------+-----+
|         lo|           hi|count|
+-----------+-------------+-----+
|EFO_0008111|   HP_0001288|    2|
|EFO_0008111|MONDO_0002253|    2|
|EFO_0008111|  EFO_0009544|    2|
|EFO_0008111|MONDO_0024298|    2|
|EFO_0008111|  EFO_0022187|    2|
| HP_0000360|MONDO_0005148|    2|
|EFO_0005741|MONDO_0005148|    2|
|EFO_0000341|MONDO_0005148|    2|
|EFO_0004872|MONDO_0005148|    2|
|EFO_0022270|MONDO_0005148|    2|
|EFO_0003765|  EFO_0008343|    2|
|EFO_0007800|  EFO_1001474|    2|
|EFO_0004329|MONDO_0015978|    2|
|EFO_0004329|MONDO_0004979|    2|
|EFO_0004329|MONDO_0002917|    2|
|EFO_0003966|MONDO_0001020|    2|
|EFO_0008595|MONDO_0001020|    2|
|EFO_0000305|  EFO_1000789|    2|
|EFO_0000305|  EFO_0004741|    2|
|EFO_0000305|  EFO_0021523|    2|
+-----------+-------------+-----+
only showing top 20 rows



In [13]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

canon = exploded.withColumn("lo", f.least("diseaseId1", "diseaseId2")).withColumn(
    "hi", f.greatest("diseaseId1", "diseaseId2")
)

# drop rows with missing rg (or rg_se, since we order on it) before dedup
canon = canon.filter(f.col("rg").isNotNull() & f.col("rg_se").isNotNull()).filter(f.col("n_snps_used") >= 100_000)

w = Window.partitionBy("lo", "hi").orderBy(f.col("rg_se").asc())

deduped = canon.withColumn("rn", f.row_number().over(w)).filter("rn == 1").drop("rn")

# sanity check: should now be zero
deduped.groupBy("lo", "hi").count().filter("count > 1").show()

+---+---+-----+
| lo| hi|count|
+---+---+-----+
+---+---+-----+



In [14]:
deduped.count()

618991

In [15]:
import numpy as np
import pandas as pd

# collect deduped pairs to the driver — one row per unique disease pair
pdf = deduped.select("lo", "hi", "rg").toPandas()

# full list of diseases, sorted for a stable, reproducible index order
diseases = sorted(set(pdf["lo"]) | set(pdf["hi"]))
idx = {d: i for i, d in enumerate(diseases)}
n = len(diseases)

matrix = np.zeros((n, n))

rows = pdf["lo"].map(idx).to_numpy()
cols = pdf["hi"].map(idx).to_numpy()
vals = pdf["rg"].to_numpy()
vals = np.clip(vals, -1.0, 1.0)  # cap rg to [-1, 1]
vals = np.nan_to_num(vals, nan=0.0)  # replace null rg with 0

matrix[rows, cols] = vals
matrix[cols, rows] = vals  # symmetric

# optional: diagonal = 1 (self rg)
np.fill_diagonal(matrix, 1.0)

In [16]:
matrix

array([[ 1.        ,  0.56661774,  0.26108634, ...,  1.        ,
        -1.        ,  1.        ],
       [ 0.56661774,  1.        ,  0.47080698, ...,  0.34594743,
        -0.01636234,  0.06494622],
       [ 0.26108634,  0.47080698,  1.        , ...,  0.13065678,
         0.01890467,  0.3636362 ],
       ...,
       [ 1.        ,  0.34594743,  0.13065678, ...,  1.        ,
        -0.1805689 ,  0.03571408],
       [-1.        , -0.01636234,  0.01890467, ..., -0.1805689 ,
         1.        ,  0.03959733],
       [ 1.        ,  0.06494622,  0.3636362 , ...,  0.03571408,
         0.03959733,  1.        ]], shape=(1114, 1114))

In [17]:
diseases

['EFO_0000195',
 'EFO_0000217',
 'EFO_0000266',
 'EFO_0000274',
 'EFO_0000275',
 'EFO_0000280',
 'EFO_0000284',
 'EFO_0000305',
 'EFO_0000318',
 'EFO_0000319',
 'EFO_0000341',
 'EFO_0000349',
 'EFO_0000373',
 'EFO_0000384',
 'EFO_0000389',
 'EFO_0000400',
 'EFO_0000464',
 'EFO_0000474',
 'EFO_0000478',
 'EFO_0000480',
 'EFO_0000537',
 'EFO_0000538',
 'EFO_0000540',
 'EFO_0000546',
 'EFO_0000555',
 'EFO_0000565',
 'EFO_0000571',
 'EFO_0000589',
 'EFO_0000612',
 'EFO_0000616',
 'EFO_0000618',
 'EFO_0000625',
 'EFO_0000649',
 'EFO_0000660',
 'EFO_0000662',
 'EFO_0000668',
 'EFO_0000676',
 'EFO_0000677',
 'EFO_0000684',
 'EFO_0000685',
 'EFO_0000701',
 'EFO_0000708',
 'EFO_0000712',
 'EFO_0000729',
 'EFO_0000731',
 'EFO_0000734',
 'EFO_0000756',
 'EFO_0000759',
 'EFO_0000768',
 'EFO_0001060',
 'EFO_0001065',
 'EFO_0001071',
 'EFO_0001072',
 'EFO_0001073',
 'EFO_0001074',
 'EFO_0001075',
 'EFO_0001357',
 'EFO_0001365',
 'EFO_0001421',
 'EFO_0001422',
 'EFO_0001645',
 'EFO_0001663',
 'EFO_00

In [18]:
labeled = pd.DataFrame(matrix, index=diseases, columns=diseases)

In [19]:
labeled

,EFO_0000195,EFO_0000217,EFO_0000266,EFO_0000274,EFO_0000275,EFO_0000280,EFO_0000284,EFO_0000305,EFO_0000318,EFO_0000319,...,OBA_2050111,OBA_2050113,OBA_2050114,OBA_2050115,OBA_2050116,OBA_2050196,OBA_2050200,OBA_2050204,OBA_2050255,OBA_2050296
EFO_0000195,1.000000,0.566618,0.261086,0.090030,0.214648,1.000000,-0.002866,-0.049937,0.214522,-0.754032,...,-0.750778,-0.116596,1.000000,-1.000000,0.235316,1.000000,-0.063584,1.000000,-1.000000,1.000000
EFO_0000217,0.566618,1.000000,0.470807,0.145949,0.009795,1.000000,0.196701,-0.005672,0.180590,-0.354969,...,0.326061,0.053266,0.065256,-0.110472,0.129744,1.000000,-0.140883,0.345947,-0.016362,0.064946
EFO_0000266,0.261086,0.470807,1.000000,0.168170,0.227077,1.000000,0.022068,-0.063462,0.090658,-0.531411,...,0.131813,-0.251512,-0.011588,-0.158178,0.061165,-0.104305,-0.067380,0.130657,0.018905,0.363636
EFO_0000274,0.090030,0.145949,0.168170,1.000000,-0.009771,0.143358,0.059510,0.340831,0.170915,-0.093273,...,0.277876,0.108172,-0.047807,0.014209,0.061131,-0.279457,-0.021341,-0.205005,-0.014962,-0.066698
EFO_0000275,0.214648,0.009795,0.227077,-0.009771,1.000000,0.171890,0.092740,0.010113,0.388129,-0.233586,...,0.231375,0.040204,0.001121,-0.039937,0.054217,0.054742,-0.010949,0.311945,0.012775,-0.020412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
OBA_2050196,1.000000,1.000000,-0.104305,-0.279457,0.054742,0.109680,-0.106221,-0.012938,-0.033578,-1.000000,...,-0.233543,-0.184014,0.673333,-0.848966,0.499101,1.000000,-0.074645,0.076274,-0.006011,0.626275
OBA_2050200,-0.063584,-0.140883,-0.067380,-0.021341,-0.010949,-0.268878,0.519235,0.003793,-0.059447,0.055231,...,0.166284,-0.122461,-0.041919,0.079442,-0.101784,-0.074645,1.000000,-0.082336,0.410770,-0.017996
OBA_2050204,1.000000,0.345947,0.130657,-0.205005,0.311945,0.010322,0.104281,-0.053542,0.715917,-0.464688,...,0.016195,0.356382,0.264313,-0.374567,0.222176,0.076274,-0.082336,1.000000,-0.180569,0.035714
OBA_2050255,-1.000000,-0.016362,0.018905,-0.014962,0.012775,-0.085174,0.611609,0.141966,-0.328651,0.000903,...,-0.170743,0.000935,0.032046,0.010417,-0.020678,-0.006011,0.410770,-0.180569,1.000000,0.039597


In [20]:
import numpy as np

vals = labeled.values

print("NaN count:", np.isnan(vals).sum())
print("Inf count:", np.isinf(vals).sum())
print("Min:", np.nanmin(vals), "Max:", np.nanmax(vals))
print("Out of [-1, 1] range:", ((vals < -1) | (vals > 1)).sum())
print("Symmetric:", np.allclose(vals, vals.T, equal_nan=True))
print("Diagonal all 1s:", np.allclose(np.diag(vals), 1.0))

NaN count: 0
Inf count: 0
Min: -1.0 Max: 1.0
Out of [-1, 1] range: 0
Symmetric: True
Diagonal all 1s: True


In [21]:
labeled.to_parquet(path_to_intermediate_data_folder + "canonical_pairwise_table/rg_processed.parquet")

In [22]:
# double check

In [23]:
import pandas as pd

labeled = pd.read_parquet(path_to_intermediate_data_folder + "canonical_pairwise_table/rg_processed.parquet")

# quick checks
print(labeled.shape)
print(labeled.index[:5], labeled.columns[:5])  # confirm disease labels loaded correctly
print(labeled.values.diagonal()[:5])  # should be all 1s
print(labeled.isna().sum().sum())  # total NaNs, should be 0
print((labeled.values < -1).sum() + (labeled.values > 1).sum())  # out-of-range count, should be 0

(1114, 1114)
Index(['EFO_0000195', 'EFO_0000217', 'EFO_0000266', 'EFO_0000274',
       'EFO_0000275'],
      dtype='object') Index(['EFO_0000195', 'EFO_0000217', 'EFO_0000266', 'EFO_0000274',
       'EFO_0000275'],
      dtype='object')
[1. 1. 1. 1. 1.]
0
0
